# Airtel Telecom Analytics — Phase 1: Synthetic Data Generation

**Tools:** Faker · SDV GaussianCopulaSynthesizer · YData Profiling  
**Output:** `airtel_raw_data/` — 4 noisy CSVs + SDV-expanded CSVs + HTML profiling reports

---
**Pipeline:**
1. Generate realistic seed data (Indian telecom context) using Faker
2. Inject controlled noise (nulls, duplicates, mixed case, type corruption)
3. Expand dataset with SDV AI synthesis (GaussianCopula)
4. Generate YData Profiling HTML reports

**Next:** `02_data_cleaning.ipynb`

In [ ]:
# Install dependencies (run once)
# !pip install sdv faker ydata-profiling pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import warnings
warnings.filterwarnings('ignore')

# Pipeline shared modules
from config import *
from utils import log, section, snapshot, reset_log, save_report

# Set all seeds for reproducibility
Faker.seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
fake = Faker('en_IN')  # Indian locale

section('AIRTEL TELECOM — Phase 1: Synthetic Data Generation')
log(f'Config: {N_CUSTOMERS} customers · {N_BILLING} billing · {N_CHURN} churn · {N_TICKETS} tickets')

In [ ]:
# ─── NOISE INJECTION ──────────────────────────────────────────────────────────

def inject_noise(df: pd.DataFrame, null_rate: float = 0.08, dupe_rate: float = 0.02,
                 seed: int = SEED) -> pd.DataFrame:
    """
    Simulate real-world data fragmentation:
      1. Random nulls in non-ID columns (null_rate % of values)
      2. Duplicate rows (dupe_rate % of rows appended)
      3. Mixed case / formatting issues in string columns (5% of values)
      4. Type corruption — numeric values get ' INR' suffix (3% of one column)

    seed is set locally so shuffled output is reproducible.
    """
    rng = np.random.RandomState(seed)  # local RNG — doesn't affect global state
    df = df.copy()

    # 1. Nulls
    non_id_cols = [c for c in df.columns if 'id' not in c.lower()]
    for col in non_id_cols:
        mask = rng.random(len(df)) < null_rate
        df.loc[mask, col] = np.nan

    # 2. Duplicates
    n_dupes = int(len(df) * dupe_rate)
    if n_dupes > 0:
        dupes = df.sample(n_dupes, replace=True, random_state=seed)
        df = pd.concat([df, dupes], ignore_index=True)

    # 3. Mixed case (5% of string values — subtle but detectable)
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        mask = rng.random(len(df)) < 0.05
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: x.upper() if isinstance(x, str) else x
        )

    # 4. Type corruption — one random numeric column gets ' INR' appended
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        col = rng.choice(num_cols)
        mask = rng.random(len(df)) < 0.03
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: f'{x} INR' if pd.notna(x) else x
        )

    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
# ─── PHASE 1: CUSTOMERS ───────────────────────────────────────────────────────

section('[1/4] Generating Customers')

customers = []
for i in range(N_CUSTOMERS):
    region    = random.choice(list(REGIONS.keys()))
    state     = random.choice(REGIONS[region])
    plan_type = random.choices(['Prepaid', 'Postpaid'], weights=[65, 35])[0]
    plan_name = random.choice(PREPAID_PLANS if plan_type == 'Prepaid' else POSTPAID_PLANS)

    customers.append({
        'customer_id':         f'AIRTEL_{str(i+1).zfill(5)}',
        'name':                fake.name(),
        'age':                 random.randint(18, 65),
        'gender':              random.choices(['Male', 'Female'], weights=[55, 45])[0],
        'region':              region,
        'state':               state,
        'plan_type':           plan_type,
        'plan_name':           plan_name,
        'plan_price':          PLAN_PRICE[plan_name],
        'tenure_months':       random.randint(1, 72),
        'contract_type':       random.choices(['Monthly', 'Annual'], weights=[70, 30])[0],
        'phone_number':        f"{random.choice([6,7,8,9])}{fake.numerify('#########')}",
        'email':               fake.email() if random.random() > 0.15 else None,
        'kyc_verified':        random.choices(['Yes', 'No'], weights=[80, 20])[0],
        'acquisition_channel': random.choice(ACQ_CHANNELS),
        'city':                fake.city(),
        'pincode':             fake.postcode(),
        'joining_date':        fake.date_between(start_date='-6y', end_date='-1m'),
        'is_active':           random.choices([1, 0], weights=[75, 25])[0],
    })

customers_df    = pd.DataFrame(customers)
customers_noisy = inject_noise(customers_df, NULL_RATE_CUSTOMERS, DUPE_RATE_CUSTOMERS)
customers_noisy.to_csv(f'{RAW_DIR}/customers_raw.csv', index=False)
snapshot(customers_noisy, 'customers_raw')
log(f'  ✓ Saved → {RAW_DIR}/customers_raw.csv')

In [ ]:
# ─── PHASE 2: BILLING ─────────────────────────────────────────────────────────

section('[2/4] Generating Billing')

# FIX: Build lookup dict once — O(1) per lookup vs O(n) row filter in original
cust_lookup   = customers_df.set_index('customer_id')[['plan_price']].to_dict('index')
cust_ids      = customers_df['customer_id'].tolist()
bill_months   = pd.date_range('2023-01-01', periods=12, freq='MS')

billing = []
for i in range(N_BILLING):
    cid      = random.choice(cust_ids)
    plan_chg = cust_lookup[cid]['plan_price']  # O(1) dict lookup
    month    = random.choice(bill_months)
    data_chg = random.choices([0, 50, 100, 150, 200], weights=[50, 20, 15, 10, 5])[0]
    call_chg = random.choices([0, 20, 45, 80],        weights=[60, 20, 15,  5])[0]
    sms_chg  = random.choices([0, 5, 10],              weights=[70, 20, 10])[0]
    subtotal = plan_chg + data_chg + call_chg + sms_chg
    taxes    = round(subtotal * 0.18, 2)
    discount = random.choices([0, 25, 50, 100], weights=[60, 20, 15, 5])[0]
    total    = round(subtotal + taxes - discount, 2)
    paid     = random.choices(['Paid', 'Unpaid', 'Partial'], weights=[75, 15, 10])[0]
    arrears  = plan_chg if paid == 'Unpaid' else (round(total * 0.5, 2) if paid == 'Partial' else 0)

    billing.append({
        'bill_id':          f'BILL_{str(i+1).zfill(6)}',
        'customer_id':      cid,
        'billing_month':    month.strftime('%Y-%m'),
        'plan_charges':     plan_chg,
        'data_charges':     data_chg,
        'call_charges':     call_chg,
        'sms_charges':      sms_chg,
        'taxes':            taxes,
        'discount_applied': discount,
        'total_amount':     total,
        'payment_status':   paid,
        'payment_date':     fake.date_between(
                                start_date=month,
                                end_date=month + pd.DateOffset(months=1)
                            ).strftime('%Y-%m-%d') if paid == 'Paid' else None,
        'payment_method':   random.choices(
                                ['UPI', 'Credit Card', 'Debit Card', 'Net Banking', 'Cash'],
                                weights=[40, 20, 20, 15, 5])[0],
        'arrears':          arrears,
    })

billing_df    = pd.DataFrame(billing)
billing_noisy = inject_noise(billing_df, NULL_RATE_BILLING, DUPE_RATE_BILLING)
billing_noisy.to_csv(f'{RAW_DIR}/billing_raw.csv', index=False)
snapshot(billing_noisy, 'billing_raw')
log(f'  ✓ Saved → {RAW_DIR}/billing_raw.csv')

In [ ]:
# ─── PHASE 3: CHURN EVENTS ────────────────────────────────────────────────────

section('[3/4] Generating Churn Events')

churn_pool = random.sample(cust_ids, min(N_CHURN, len(cust_ids)))
churn_data = []

for i, cid in enumerate(churn_pool):
    churn_dt  = fake.date_between(start_date='-18m', end_date='today')
    reason    = random.choice(CHURN_REASONS)
    retained  = random.choices(['Yes', 'No'], weights=[30, 70])[0]
    reactivated = random.choices(['Yes', 'No'], weights=[20, 80])[0] if retained == 'No' else 'No'

    churn_data.append({
        'churn_id':            f'CHURN_{str(i+1).zfill(5)}',
        'customer_id':         cid,
        'churn_date':          churn_dt.strftime('%Y-%m-%d'),
        'churn_reason':        reason,
        'competitor_moved_to': random.choice(COMPETITORS) if reason != 'Payment Default' else '',
        'churn_type':          'Involuntary' if reason == 'Payment Default' else 'Voluntary',
        'was_retained':        retained,
        'retention_offer':     random.choice(['Cashback 100', 'Free Month', 'Plan Upgrade', '']) if retained == 'Yes' else '',
        'reactivated':         reactivated,
        'reactivation_date':   fake.date_between(
                                   start_date=churn_dt, end_date='today'
                               ).strftime('%Y-%m-%d') if reactivated == 'Yes' else None,
        'lifetime_value':      round(random.uniform(500, 15000), 2),
    })

churn_df    = pd.DataFrame(churn_data)
churn_noisy = inject_noise(churn_df, NULL_RATE_CUSTOMERS, DUPE_RATE_CUSTOMERS)
churn_noisy.to_csv(f'{RAW_DIR}/churn_events_raw.csv', index=False)
snapshot(churn_noisy, 'churn_raw')
log(f'  ✓ Saved → {RAW_DIR}/churn_events_raw.csv')

In [ ]:
# ─── PHASE 4: SUPPORT TICKETS ─────────────────────────────────────────────────

section('[4/4] Generating Support Tickets')

tickets = []
for i in range(N_TICKETS):
    cid     = random.choice(cust_ids)
    created = fake.date_between(start_date='-18m', end_date='today')
    resolved = random.choices([True, False], weights=[75, 25])[0]

    tickets.append({
        'ticket_id':             f'TKT_{str(i+1).zfill(6)}',
        'customer_id':           cid,
        'created_date':          created.strftime('%Y-%m-%d'),
        'issue_category':        random.choice(ISSUE_CATEGORIES),
        'priority':              random.choices(['Low','Medium','High','Critical'], weights=[25,35,30,10])[0],
        'channel':               random.choice(CHANNELS),
        'assigned_agent':        f'AGT_{str(random.randint(1,25)).zfill(2)}',
        'resolution_date':       fake.date_between(
                                     start_date=created,
                                     end_date=created + pd.DateOffset(days=10)
                                 ).strftime('%Y-%m-%d') if resolved else None,
        'resolution_status':     random.choices(['Resolved','Unresolved','Escalated'], weights=[70,15,15])[0] if resolved else 'Unresolved',
        'csat_score':            random.randint(1, 5) if resolved else None,
        'escalated':             random.choices([True, False], weights=[15, 85])[0],
        'first_contact_resolved':random.choices([True, False], weights=[60, 40])[0] if resolved else False,
        'handle_time_mins':      random.randint(3, 90),
    })

tickets_df    = pd.DataFrame(tickets)
tickets_noisy = inject_noise(tickets_df, NULL_RATE_CUSTOMERS, DUPE_RATE_CUSTOMERS)
tickets_noisy.to_csv(f'{RAW_DIR}/support_tickets_raw.csv', index=False)
snapshot(tickets_noisy, 'tickets_raw')
log(f'  ✓ Saved → {RAW_DIR}/support_tickets_raw.csv')

In [ ]:
# ─── SDV AI SYNTHESIS ─────────────────────────────────────────────────────────

section('SDV AI Synthesis (GaussianCopula)')

try:
    from sdv.single_table import GaussianCopulaSynthesizer
    from sdv.metadata import SingleTableMetadata

    def sdv_synthesize(df: pd.DataFrame, name: str, n_rows: int):
        log(f'  → Training SDV on {name} ({len(df)} seed rows)...')
        df_sdv = df.select_dtypes(include=['number', 'object']).copy()
        df_sdv = df_sdv.dropna(thresh=int(len(df_sdv.columns) * 0.5))
        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(df_sdv)
        synth = GaussianCopulaSynthesizer(metadata)
        synth.fit(df_sdv)
        synthetic = synth.sample(num_rows=n_rows)
        out = f'{RAW_DIR}/{name}_sdv_synthetic.csv'
        synthetic.to_csv(out, index=False)
        log(f'  ✓ {n_rows} synthetic rows → {out}')

    sdv_synthesize(customers_df, 'customers', SDV_CUSTOMERS_ROWS)
    sdv_synthesize(billing_df,   'billing',   SDV_BILLING_ROWS)
    sdv_synthesize(churn_df,     'churn',     SDV_CHURN_ROWS)
    log('  ✅ SDV synthesis complete!')

except ImportError:
    log('  ⚠ SDV not installed — skipping. Run: pip install sdv')

In [ ]:
# ─── YDATA PROFILING ──────────────────────────────────────────────────────────

section('YData Profiling Reports')
log(f'  minimal={YDATA_MINIMAL} (set YDATA_MINIMAL=False in config.py for full reports)')

try:
    from ydata_profiling import ProfileReport

    datasets = {
        'customers': customers_noisy,
        'billing':   billing_noisy,
        'churn':     churn_noisy,
        'tickets':   tickets_noisy,
    }

    for name, df in datasets.items():
        log(f'  → Profiling {name}...')
        profile = ProfileReport(
            df,
            title=f'Airtel {name.capitalize()} — Data Quality Report',
            minimal=YDATA_MINIMAL,   # FIX: was hardcoded False — very slow on large data
        )
        report_path = f'{RAW_DIR}/{name}_profile_report.html'
        profile.to_file(report_path)
        log(f'  ✓ Report → {report_path}')

    log('  ✅ All profiling reports done!')

except ImportError:
    log('  ⚠ ydata-profiling not installed — skipping. Run: pip install ydata-profiling')

In [ ]:
# ─── SUMMARY ──────────────────────────────────────────────────────────────────

section('Phase 1 Complete')
log(f'''
  Raw (noisy) CSVs → {RAW_DIR}/
    customers_raw.csv        ({len(customers_noisy):,} rows)
    billing_raw.csv          ({len(billing_noisy):,} rows)
    churn_events_raw.csv     ({len(churn_noisy):,} rows)
    support_tickets_raw.csv  ({len(tickets_noisy):,} rows)

  NEXT → Run 02_data_cleaning.ipynb
''')

save_report(f'{RAW_DIR}/phase1_report.txt')